# Bölüm 17: Dağıtım Seçenekleri
**İlk LLM'ini Oluştur — Bölüm 17: Dağıtım Seçenekleri**

Bu defteri Jupyter’da açabilir veya Colab’a yerel `.ipynb` dosyasını yükleyebilirsin. Hücreler yalnızca kaynak kodu ve terminal komutlarını gösterir. Yardımcı dosya veya kimlik bilgisi gerekmez; defter paket kurmaz, istek göndermez ve dağıtım yapmaz.

Modal örneklerini ayrıca kendi terminalinde çalıştırmayı seçersen uzak fonksiyonlar ücret doğurabilir.

In [ ]:
print("İsteğe bağlı: yerel terminalde Python 3.11 sanal ortamı kullan.")
print("python -m pip install modal fastapi uvicorn requests pydantic httpx")
print("modal setup  # CLI oturumu açar; web erişimi için proxy token’ı ayrıca gerekir.")

## Bölüm 17.1: Neden Modal?

**Sunucusuz (Serverless) Nedir?**

| Geleneksel Sunucu | Sunucusuz (Modal) |
|-------------------|-------------------|
| 7/24 bir bilgisayar kiralarsın | Kod yalnızca gerektiğinde çalışır |
| Boştayken bile ödeme yaparsın | Hesaplama saniyesi başına ödeme yaparsın |
| Güncellemeleri, güvenliği sen yönetirsin | Platform sunucuyu korur; uygulamaya erişimi yine sen yönetirsin |
| Sabit kapasite | Otomatik ölçekleme |

**Analoji:** Geleneksel barındırma, araba sahibi olmak gibidir. Sunucusuz, taksi kullanmak gibidir; sadece yolculuk yaptığında ödersin.

**LLM'ler için Neden Modal?**
- Yerleşik GPU desteği (T4'ten H100'e)
- Ayda 30$ ücretsiz kredi (~37 saat L4 GPU süresi; T4'te ~50 saat)
- Python'a özgü (Docker yok, YAML yok)
- Sıfıra ölçekler (boştayken ücret yok)

## Bölüm 17.2: Merhaba Modal

En basit Modal uygulaması. Bunu `hello.py` olarak kaydet ve `modal run hello.py` ile çalıştır:

In [ ]:
# Bunu hello.py olarak kaydet
hello_modal_code = '''
import modal

app = modal.App("hello-world")

@app.function()
def hello(name: str) -> str:
    return f"Merhaba, {name}!"

@app.local_entrypoint()
def main():
    result = hello.remote("World")
    print(result)
'''

print("Bu kodu 'hello.py' olarak kaydet:")
print(hello_modal_code)
print("\nArdından çalıştır: modal run hello.py")

**Ne oldu?**

1. `modal.App()` uygulamanı oluşturur
2. `@app.function()` Modal'ın bulutunda çalışacak kodu işaretler
3. `hello.remote()` fonksiyonu uzaktan çağırır
4. Modal bir container başlattı, kodunu çalıştırdı, sonucu döndürdü

## Bölüm 17.3: GPU Desteği Ekleme

Tek bir parametre GPU desteği ekler:

In [ ]:
# Bunu gpu_test.py olarak kaydet
gpu_test_code = '''
import modal

app = modal.App("gpu-test")

# torch, container görüntüsüne kurulmalı (sonraki bölüm görüntüleri anlatır)
image = modal.Image.debian_slim(python_version="3.11").pip_install("torch")

@app.function(image=image, gpu="L4")  # Hepsi bu!
def check_gpu():
    import torch
    if torch.cuda.is_available():
        device = torch.cuda.get_device_name(0)
        return f"GPU available: {device}"
    return "No GPU found"

@app.local_entrypoint()
def main():
    print(check_gpu.remote())
'''

print("Bu kodu 'gpu_test.py' olarak kaydet:")
print(gpu_test_code)
print("\nArdından çalıştır: modal run gpu_test.py")
print("\nBeklenen çıktı: GPU available: NVIDIA L4")

**Tek bir parametre.** CUDA kurulumu yok, sürücü yönetimi yok, Docker imajları yok. Modal tüm GPU yığınını halleder.

## Bölüm 17.4: Bağımlılıkları Yükleme

LLM'in kütüphanelere ihtiyaç duyar. Modal, container ortamını Python'da tanımlamana izin verir:

In [ ]:
# Bunu with_deps.py olarak kaydet
with_deps_code = '''
import modal

# Container imajını tanımla
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "torch",
    "transformers",
    "accelerate",
)

app = modal.App("with-deps")

@app.function(image=image, gpu="T4")
def generate_text():
    from transformers import pipeline
    
    generator = pipeline("text-generation", model="gpt2", device=0)
    result = generator("The meaning of life is", max_length=50)
    return result[0]["generated_text"]

@app.local_entrypoint()
def main():
    print(generate_text.remote())
'''

print("Bu kodu 'with_deps.py' olarak kaydet:")
print(with_deps_code)
print("\nArdından çalıştır: modal run with_deps.py")

`image` parametresi Modal'a neyin yükleneceğini söyler. İlk çalıştırma daha uzun sürer (imaj oluşturulurken), ancak sonraki çalıştırmalar önbelleğe alınmış imajı yeniden kullanır.

## Bölüm 17.5: FastAPI ile Web Uç Noktaları

CPU örneğinde `/health` sabit bir durum, `/hello/{name}` bir selamlama döndürür. Ücretli bir fonksiyonu yalnızca URL’sini bilen herkes başlatamamalı. Bu nedenle belgeler dahil tüm ASGI uygulamasını Modal’ın proxy katmanında koru.

Önce Modal’da **Settings → Proxy auth tokens** bölümünden veya özel bir terminalde `modal workspace proxy-tokens create --name chapter17` ile proxy token’ı oluştur. Kimliği ve yalnızca bir kez gösterilen sırrı parola yöneticine kaydet. Ayrı bir deneme ortamı kullan; rol tabanlı izinler açıksa erişimi bu ortamla sınırla. Token’a uygulamanın adını vermek onu tek URL ile sınırlamaz. Bu çifti deftere, tarayıcı uygulamasına veya ekran görüntüsüne koyma.

In [ ]:
# Bunu simple_api.py olarak kaydet
simple_api_code = '''
import modal
from fastapi import FastAPI

app = modal.App("my-api")
web_app = FastAPI()

# Bağımlılıklar dizüstünde değil, isteği alan container’da gerekir.
image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi==0.163.2",
    "uvicorn==0.54.0",
    "httpx==0.28.1",
    "pydantic==2.14.3",
)

@web_app.get("/health")
def health():
    return {"status": "healthy"}

@web_app.get("/hello/{name}")
def hello(name: str):
    return {"message": f"Merhaba, {name}!"}

@app.function(image=image)
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    return web_app
'''

print("Bu kodu 'simple_api.py' olarak kaydet:")
print(simple_api_code)
print("\nArdından çalıştır: modal deploy simple_api.py")
print("\nŞu şekilde bir URL alacaksın: https://your-workspace--my-api-serve.modal.run")

**İsteğe bağlı terminal denemesi:** `SIMPLE_URL`, kendi `my-api` dağıtımının verdiği HTTPS adresi olmalıdır. `MODAL_KEY` ve `MODAL_SECRET` değişkenlerini özel Bash oturumuna parola yöneticinden aktar. Kabuk izlemeyi açma; değerleri komut geçmişine yazma.

```bash
curl --max-time 30 --header @- "$SIMPLE_URL/health" <<EOF
Modal-Key: $MODAL_KEY
Modal-Secret: $MODAL_SECRET
EOF
modal app stop my-api
```

Beklenen yanıt `{"status":"healthy"}`. `/hello/World` de aynı iki başlığı ister. Adresi doğrudan tarayıcıda açmak geçerli bir sağlık kontrolü değildir: proxy isteği fonksiyon başlamadan reddeder.

## Bölüm 17.6: Eksiksiz LLM Servisi

Bu servis Qwen3.5-2B’yi L4 üzerinde çalıştırır. Proxy kimlik doğrulaması container başlamadan önce yapılır. `/health` üretim yapmadan vLLM’i yoklar; `/chat` ayrıca FastAPI içinde ayrı bir uygulama token’ı ister. Geçerli proxy bilgileri olan biri, `/chat` daha sonra 401 dönse bile GPU’yu uyandırabilir.

In [ ]:
# Bunu llm_service.py olarak kaydet
# (Kitapla aynı servis: Bölüm 16 korkulukları + vllm serve alt süreci.
#  vLLM/Modal belgelerine karşı API doğrulamalı 2026-11; dağıtım Modal hesabı + GPU gerektirir.)
llm_service_code = '''
import modal
from fastapi import FastAPI, HTTPException, Request, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.responses import JSONResponse
from pydantic import BaseModel, field_validator
import os
from collections import defaultdict
import threading
import uuid
from datetime import datetime, timedelta
import subprocess
import time
import requests

MODEL = "Qwen/Qwen3.5-2B"
# Note: no trust_remote_code needed. Current models like Qwen3.5 are
# natively supported by vLLM; only enable it for repos whose custom
# code you have reviewed, since it executes code from the model repo.

image = modal.Image.debian_slim(python_version="3.11").pip_install(
    "fastapi",
    "vllm==0.34.0",  # Pin for reproducibility (current stable, checked 2026-11)
    "torch",
    "transformers",
    "requests",
)

app = modal.App("my-llm-service")
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)
web_app = FastAPI(title="My LLM API", version="1.0.0")

# Declaring the bearer scheme gives Swagger's "Authorize" button and marks
# protected routes in the docs; require_token below does the actual check.
bearer_scheme = HTTPBearer(auto_error=False)

# ===== Chapter 16 guardrails, plus auth and token budgets =====
MAX_PROMPT_CHARS = 4000  # cheap first filter; the real budget is in tokens
API_TOKEN = os.environ.get("API_TOKEN", "")  # set via Modal secret, see below
MAX_INPUT_TOKENS = 2048  # templated input budget (with the output cap, fits 8192)
MAX_OUTPUT_TOKENS = 512  # istek başına üretim sınırı; toplam harcama sınırı değil
RATE_LIMIT = 20
RATE_WINDOW = timedelta(minutes=1)
request_counts: dict[str, list[datetime]] = defaultdict(list)
_rate_lock = threading.Lock()

def require_token(credentials: HTTPAuthorizationCredentials | None = Depends(bearer_scheme)):
    """Bearer token kimlik doğrulaması: paylaşılan GPU uç noktası dünyaya açık olmamalı."""
    if not API_TOKEN or credentials is None or credentials.credentials != API_TOKEN:
        raise HTTPException(401, "Missing or invalid token")

class ChatRequest(BaseModel):
    message: str
    max_tokens: int = 256

    @field_validator("message")
    @classmethod
    def not_empty(cls, v):
        if not v.strip():
            raise ValueError("Message cannot be empty")
        if len(v) > MAX_PROMPT_CHARS:
            raise ValueError(f"Message too long (max {MAX_PROMPT_CHARS} chars)")
        return v

    @field_validator("max_tokens")
    @classmethod
    def cap_output(cls, v):
        if v < 1:
            raise ValueError("max_tokens must be positive")
        # Clamp instead of reject: clients asking for more get the cap, not an error
        return min(v, MAX_OUTPUT_TOKENS)

class ChatResponse(BaseModel):
    response: str
    latency_ms: float

def check_rate_limit(request: Request):
    client_ip = request.client.host
    now = datetime.now()
    # Check-and-update under one lock: these endpoints run in FastAPI's
    # threadpool, so without it two concurrent requests could both read
    # len == RATE_LIMIT - 1 and both slip through.
    with _rate_lock:
        recent = [t for t in request_counts[client_ip] if now - t < RATE_WINDOW]
        if len(recent) >= RATE_LIMIT:
            request_counts[client_ip] = recent
            raise HTTPException(429, "Too many requests. Please wait a minute.")
        recent.append(now)
        request_counts[client_ip] = recent

def log_request(request_id, input_tokens, output_tokens, latency_ms, status):
    # Allowlisted metadata only: never log prompts, responses, or raw
    # exception text. Modal captures stdout.
    import json
    print(json.dumps({"ts": datetime.now().isoformat(),
                      "request_id": request_id,
                      "input_tokens": input_tokens,
                      "output_tokens": output_tokens,
                      "latency_ms": round(latency_ms, 2),
                      "status": status}))

# ===== vLLM server management (one per container) =====
_server = None
_tokenizer = None

def start_vllm():
    """Start vllm serve once per container and wait until it is ready."""
    global _server, _tokenizer
    if _server is None:
        _server = subprocess.Popen([
            "vllm", "serve", MODEL,
            "--host", "127.0.0.1",  # Dışarıya yalnızca korunan ASGI uygulaması açılır.
            "--port", "8000",
            "--dtype", "bfloat16",  # L4 destekler, T4 desteklemez.
            "--max-model-len", "8192",       # cap context: keep KV cache small on L4's 24GB
            "--gpu-memory-utilization", "0.90",
        ])
        # Wait for the server's own cheap /health endpoint
        for _ in range(120):
            try:
                requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("vLLM server did not become ready")
        from transformers import AutoTokenizer
        _tokenizer = AutoTokenizer.from_pretrained(MODEL)
    return _tokenizer

@web_app.get("/health")
def health():
    """Cheap probe: ask the local vLLM server, no generation."""
    try:
        requests.get("http://localhost:8000/health", timeout=2).raise_for_status()
        return {"status": "healthy", "model": "Qwen3.5-2B"}
    except Exception:
        # 503, not 200: monitoring should see a degraded service as down
        return JSONResponse(status_code=503,
                            content={"status": "degraded", "model": "Qwen3.5-2B"})

@web_app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest, http_request: Request,
         _auth: None = Depends(require_token)):
    """Generate a response: template the prompt, let vLLM do the inference."""
    check_rate_limit(http_request)
    request_id = uuid.uuid4().hex[:12]
    start = time.time()
    try:
        tokenizer = start_vllm()
        # Instruct modeller ham metin değil, kendi sohbet şablonlarını bekler.
        # apply_chat_template mesajı modelin özel token'larıyla sarar.
        messages = [{"role": "user", "content": request.message}]
        # Real token budget: count the templated input with the same
        # tokenizer vLLM uses. Characters are not tokens. With one
        # conversation, apply_chat_template returns a dict-like batch
        # whose "input_ids" is the flat token list; len() of the batch
        # itself would count keys (2), not tokens.
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True)["input_ids"]
        if len(input_ids) > MAX_INPUT_TOKENS:
            raise HTTPException(
                422, f"Message too long ({len(input_ids)} tokens; "
                     f"max {MAX_INPUT_TOKENS} input tokens)")
        # Forward the SAME ids we just counted: /v1/completions accepts
        # a token-id list as "prompt", so vLLM cannot re-tokenize the
        # text into something different from what we budgeted.
        r = requests.post(
            "http://localhost:8000/v1/completions",
            json={"model": MODEL, "prompt": input_ids,
                  "max_tokens": request.max_tokens, "temperature": 0.7},
            timeout=120,
        )
        r.raise_for_status()
        body = r.json()
        text = body["choices"][0]["text"].strip()
        latency_ms = (time.time() - start) * 1000
        log_request(request_id, len(input_ids),
                    body.get("usage", {}).get("completion_tokens", 0),
                    latency_ms, 200)
        return ChatResponse(response=text, latency_ms=round(latency_ms, 2))
    except HTTPException as e:
        log_request(request_id, 0, 0, (time.time() - start) * 1000, e.status_code)
        raise
    except requests.HTTPError as e:
        # vLLM itself rejected the request (e.g. context overflow): pass
        # 4xx through with a sanitized message, never the backend's text.
        status = e.response.status_code if e.response is not None else 502
        if 400 <= status < 500:
            log_request(request_id, 0, 0, (time.time() - start) * 1000, status)
            raise HTTPException(status, "The model rejected this request. "
                                        "Try a shorter message.")
        log_request(request_id, 0, 0, (time.time() - start) * 1000, 502)
        raise HTTPException(502, "Inference backend error. Please try again.")
    except Exception:
        log_request(request_id, 0, 0, (time.time() - start) * 1000, 500)
        raise HTTPException(500, "Generation failed. Please try again.")

@app.function(
    image=image,
    gpu="L4",
    timeout=600,
    scaledown_window=300,  # Keep warm for 5 minutes
    volumes={"/root/.cache/huggingface": model_cache},  # model downloads once
    secrets=[modal.Secret.from_name("llm-api-token")],  # provides API_TOKEN
)
@modal.concurrent(max_inputs=10)  # Safe: the vLLM server batches concurrent requests
@modal.asgi_app(requires_proxy_auth=True)
def serve():
    start_vllm()  # Load once when the container starts, reuse for every request
    return web_app

'''

print("Bu kodu 'llm_service.py' olarak kaydet:")
print(llm_service_code)

In [ ]:
# check_service.py olarak kaydet. Defter bu istekleri göndermez.
client_code = '''
import os
import requests

URL = os.environ["LLM_URL"].rstrip("/")  # kendi dağıtımının verdiği HTTPS adresi
if not URL.startswith("https://"):
    raise ValueError("LLM_URL bir HTTPS adresi olmalıdır")
proxy = {"Modal-Key": os.environ["MODAL_KEY"], "Modal-Secret": os.environ["MODAL_SECRET"]}
chat = {**proxy, "Authorization": "Bearer " + os.environ["API_TOKEN"]}

# Önce dış katman: başlıksız bu olumsuz kontrol GPU’yu uyandırmamalı.
r = requests.get(f"{URL}/health", timeout=30, allow_redirects=False)
assert r.status_code == 401, f"Expected proxy denial, got {r.status_code}"

r = requests.get(f"{URL}/health", headers=proxy, timeout=300, allow_redirects=False)
assert r.status_code == 200 and r.json()["status"] == "healthy", f"Readiness: {r.status_code}"

def ask(headers, payload):
    return requests.post(f"{URL}/chat", headers=headers, json=payload,
                         timeout=150, allow_redirects=False)

r = ask(proxy, {"message": "hi"})  # proxy izni, uygulama izni değildir
assert r.status_code == 401, f"Expected app denial, got {r.status_code}"
r = ask({**proxy, "Authorization": "Bearer wrong-token"}, {"message": "hi"})
assert r.status_code == 401, f"Expected wrong bearer denial, got {r.status_code}"
r = ask(chat, {"message": " "})
assert r.status_code == 422, f"Expected validation error, got {r.status_code}"
r = ask(chat, {"message": "What is Python?", "max_tokens": 64})
assert r.status_code == 200 and r.json()["response"].strip(), f"Chat: {r.status_code}"
print("Proxy, hazırlık, uygulama token’ı, doğrulama ve sohbet kontrolleri başarılı.")
'''
print(client_code)


### İlk Çalıştırma Kontrol Listesi

Bu adımlar **isteğe bağlıdır** ve yalnızca özel terminalinde çalıştırılır. Canlı GPU dağıtımı yapılmış bir test sonucu değildir.

1. Hedef Modal ortamının kontrol panelinde `llm-api-token` adlı sırrı oluştur. `API_TOKEN` alanına uzun, rastgele bir değer ver. Bu sır yalnızca GPU uygulamasına gider. Proxy token çifti ayrı bir kimlik bilgisidir; CLI kimlik bilgilerini kullanma.
2. Önce fatura uyarısı kur. Ücretlendirmeyi kabul ediyorsan `modal deploy llm_service.py` çalıştır. `LLM_URL` için **kendi** dağıtımının verdiği HTTPS adresini kullan. `API_TOKEN`, `MODAL_KEY` ve `MODAL_SECRET` değerlerini istemci sürecine parola yöneticinden aktar; dosyaya, URL’ye veya defter çıktısına yazma.
3. `python check_service.py` çalıştır. Başlıksız `/health` isteğini proxy 401 ile reddetmeli; proxy çiftiyle hazır servisten 200 gelmelidir. Kalıcı 503 için `modal app logs my-llm-service` ile vLLM başlangıcını incele. Geçerli bir sonda, token üretmese de GPU’yu uyandırabilir; sık yoklama.
4. Proxy doğru, uygulama token’ı eksik veya yanlışsa `/chat` uygulamadan 401 almalıdır. İkisi de doğruysa boş mesaj için 422, geçerli mesaj için 200 ve boş olmayan yanıt beklenir. İstemci kendi zaman aşımını uygular; `timeout=600` Modal fonksiyonunun sınırıdır. Gizli başlıkları başka adrese taşımamak için istemci yönlendirmeleri izlemez.
5. `/docs` ve `/openapi.json` da proxy arkasındadır. Çıplak tarayıcı URL’si yerine iki proxy başlığını iletebilen bir istemci kullan. `/chat` için `Authorization` uygulamanın bearer token’ına ayrılmıştır.
6. Bir kontrol başarısız olsa bile deneme bitince `modal app stop my-llm-service` çalıştır. Panelde etkin container kalmadığını kontrol et; artık gerekmeyen deneme token’larını iptal et.

**Dağıtım yapmadan bitir:** Bölüm 16’daki HTTP taklitli kontrolleri çalıştır. Yazdırılan istemcide hangi isteğin container başlamadan, hangisinin uygulama içinde reddedileceğini belirle. Yerel `TestClient`, uygulamanın yetkilendirmesini ve doğrulamasını sınar; Modal proxy kabulünü, GPU hazırlığını veya soğuk başlatma maliyetini kanıtlamaz.


## Bölüm 17.7: Gizli anahtarlar ve kalıcı depolama birimleri

### Gizli anahtar oluşturma

Hedef ortamın Modal kontrol panelinde `my-secrets` adlı bir sır ve içinde `API_KEY` alanı oluştur. Kodda yalnızca sırrın adına başvur; değerini yazdırma. Proxy çifti bu fonksiyona değil güvenilir istemciye aittir.

### Kodda gizli anahtar kullanma

In [ ]:
# Kodunda sırları kullanma
secrets_example = '''
@app.function(secrets=[modal.Secret.from_name("my-secrets")])
def with_secrets():
    import os
    api_key = os.environ["API_KEY"]
    # Anahtarı güvenli bir şekilde kullan...
'''

print("Gizli anahtar kullanımı:")
print(secrets_example)

In [ ]:
# Modelleri önbelleğe almak için kalıcı depolama birimlerini kullanma
volumes_example = '''
# Model önbelleği için bir kalıcı depolama birimi oluştur
model_cache = modal.Volume.from_name("model-cache", create_if_missing=True)

@app.function(
    gpu="L4",
    volumes={"/root/.cache/huggingface": model_cache},
)
def with_cache():
    # Modeller bir kez indirilir, sonra kalıcı depolama biriminde önbelleğe alınır
    pass
'''

print("Modelleri önbelleğe almak için kalıcı depolama birimlerini kullanma:")
print(volumes_example)
print("\nİlk istek modeli indirir. Sonraki istekler önbelleği kullanır.")

## Bölüm 17.8: GPU Seçenekleri

| Model Boyutu | Önerilen GPU | Maliyet/Saat |
|--------------|--------------|-------------|
| < 3B parametre (FP16/kuantize) | T4 (16GB) | $0.59 |
| < 3B parametre (BF16, örn. Qwen3.5-2B) | L4 (24GB) | $0.80 |
| 3-8B parametre | A10 (24GB) | $1.10 |
| 8-30B parametre | A100-40GB | $2.50 |
| 30B+ parametre | A100-80GB / H100 | $4-8 |

(Fiyatlar 2026-11'de kontrol edildi; modal.com/pricing adresinden doğrula.)

**L4 ile başla.** Qwen3.5-2B'nin ağırlıkları BF16'dır ve vLLM 0.34, Ampere'den (compute capability 8.0) eski GPU'larda BF16'yı reddeder: T4 7.5'tir ve başlangıçta çıkar; L4 (8.9) ve A10 (8.6) bu eşiği geçer. T4, daha küçük FP16 ya da kuantize modeller için hâlâ çalışır. Yalnızca gerektiğinde yükselt.

In [ ]:
# GPU seçimi örnekleri
print("GPU seçimi tek bir parametredir:")
print()
print('@app.function(gpu="T4")      # 16GB, $0.59/saat - yalnızca FP16 veya kuantize modeller (BF16 yok)')
print('@app.function(gpu="L4")      # 24GB, $0.80/saat - Qwen3.5-2B için seçimimiz')
print('@app.function(gpu="A10")     # 24GB, $1.10/saat - aynı bellek, daha hızlı hesaplama')
print('@app.function(gpu="A100")    # Yüksek performans')
print('@app.function(gpu="H100")    # Maksimum güç')
print('@app.function(gpu="A100:2")  # İki A100')

## Bölüm 17.9: Dağıtımını Yönetme

### Logları Görüntüleme

```bash
modal app logs my-llm-service
```

Ya da [modal.com](https://modal.com) adresindeki kontrol panelini kullan.

### Uygulamanı Güncelleme

```bash
# Sadece yeniden dağıt
modal deploy llm_service.py
```

Kesinti olmaz. Modal, sürekli güncellemeleri otomatik olarak halleder.

### Maliyet Kontrolü

- Boştaki container'ları kapatmak için `scaledown_window` kullan
- L4 ile başla, yalnızca gerektiğinde yükselt
- Modal kontrol panelinde harcama uyarıları kur; `modal app stop my-llm-service` her şeyi kapatır
- Harcamaları Modal kontrol panelinde izle
- 4.000 karakter, 2.048 şablonlanmış girdi token’ı ve 512 çıktı token’ı sınırları istek başınadır. IP başına dakikada 20 istek sınırı her container için ayrıdır ve kapanınca sıfırlanır. Bunlar veya fatura uyarısı toplam harcamayı sınırlamaz. Paylaşılan bir hizmet için kullanıcı kotalarını ortak bir depoda atomik olarak uygula.

## Özet

Şunları öğrendin:

1. **Fonksiyonları dağıtmak** `@app.function()` ile Modal'a
2. **GPU desteği eklemek** tek bir parametreyle: `gpu="L4"`
3. **Bağımlılıkları yüklemek** `modal.Image` kullanarak
4. **Tüm web rotalarını korumak:** `@modal.asgi_app(requires_proxy_auth=True)` ile proxy, container başlamadan önce kimliği doğrular.
5. **Gizli anahtarları güvenle saklamak:** `modal.Secret` ile
6. **Modelleri önbelleğe almak** `modal.Volume` ile
7. **Maliyetleri kontrol etmek** boşta kalma zaman aşımları ve izleme ile

**Arayüzü paylaş, kimlik bilgilerini değil.** URL tek başına erişim vermez. Çalışma alanının proxy çiftini güvenilir bir istemcide veya sunucuda tut. Ayrı bearer token’ını yalnızca üretim yetkisi olan istemcilere ver. Son kullanıcıları kendi kimlik doğrulamalı geçidinden geçir.

**Hatırlanması gereken komutlar:**
```bash
pip install modal      # Kur
modal setup            # Kimlik doğrula
modal run file.py      # Bir kez çalıştır
modal deploy file.py   # Kalıcı olarak dağıt
modal app logs name    # Logları görüntüle
```